# NFL Big Data Bowl 2026 - ST-GNN Transformer Training

This notebook trains a state-of-the-art Spatial-Temporal Graph Neural Network with Transformer encoder for NFL trajectory prediction.

## Model Architecture:
- **Node Encoder**: MLP encoding player features
- **Edge Encoder**: MLP encoding spatial relationships
- **Graph Attention Layers**: GATv2-style message passing
- **Evoformer Pairwise Updates**: AlphaFold-inspired pairwise interactions
- **Temporal Transformer**: Multi-head attention for temporal dynamics
- **Prediction Head**: Outputs (Δx, Δy) trajectory

## Training:
- Optimizer: AdamW
- Scheduler: Cosine Annealing
- Loss: MSE + acceleration smoothness
- Batch size: 32
- GPU: Auto-selection (optimized for dual RTX 5090)


In [2]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
from collections import defaultdict

# Import config
import config

# Import preprocessing
from preprocessing import load_sequences

# Import model and utilities
from src.utils import NFLTrajectoryDataset, collate_fn

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    if torch.cuda.device_count() > 1:
        print(f"Multiple GPUs detected: {torch.cuda.device_count()}")
print()


Using device: cuda
GPU: NVIDIA GeForce RTX 5090
CUDA Version: 13.0
Multiple GPUs detected: 2



## 1. Load Sequences


In [3]:
print("Loading sequences...")
print("-" * 60)

sequences = load_sequences()

# Get unique game_ids for train/val split
game_ids = sorted(set(key[0] for key in sequences.keys()))
n_games = len(game_ids)
cutoff_idx = int(0.8 * n_games)
train_games = set(game_ids[:cutoff_idx])
val_games = set(game_ids[cutoff_idx:])

# Split sequences
train_sequences = {k: v for k, v in sequences.items() if k[0] in train_games}
val_sequences = {k: v for k, v in sequences.items() if k[0] in val_games}

print(f"Total sequences: {len(sequences):,}")
print(f"Train sequences: {len(train_sequences):,}")
print(f"Validation sequences: {len(val_sequences):,}")
print()


Loading sequences...
------------------------------------------------------------
✓ Loaded 46,045 sequences from: /home/joshua/nfl-bdb/data/processed/sequences.pkl
Total sequences: 46,045
Train sequences: 37,009
Validation sequences: 9,036



## 2. Load Input DataFrame for Graph Construction


In [4]:
from preprocessing import load_all_train_inputs, normalize_play_direction, add_features

print("Loading input data for graph construction...")
print("-" * 60)

# Load and preprocess input data (needed for building graphs with all players)
input_df = load_all_train_inputs(config.DATA_PATH)
input_df = normalize_play_direction(input_df)
input_df = add_features(input_df)

print(f"✓ Loaded {len(input_df):,} input rows")
print(f"  Columns: {len(input_df.columns)}")
print()


Loading input data for graph construction...
------------------------------------------------------------
✓ Loaded 4,880,579 input rows
  Columns: 55



## 3. Create DataLoaders


In [5]:
# Dataset parameters
BATCH_SIZE = 32
MAX_TRAJECTORY_LENGTH = 100
MAX_PLAYERS = 22

print("Creating datasets...")
print("-" * 60)

train_dataset = NFLTrajectoryDataset(
    train_sequences,
    all_input_df=input_df,
    max_players=MAX_PLAYERS,
    max_trajectory_length=MAX_TRAJECTORY_LENGTH
)

val_dataset = NFLTrajectoryDataset(
    val_sequences,
    all_input_df=input_df,
    max_players=MAX_PLAYERS,
    max_trajectory_length=MAX_TRAJECTORY_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True if torch.cuda.is_available() else False
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print()


Creating datasets...
------------------------------------------------------------
Train batches: 1157
Validation batches: 283



## 4. Build Model


In [8]:
# Model hyperparameters
HIDDEN_DIM = 256
NUM_GRAPH_LAYERS = 3
NUM_TEMPORAL_LAYERS = 4
NUM_HEADS = 8
DROPOUT = 0.1
NODE_INPUT_DIM = 13  # x, y, vx, vy, s, a, dist_ball, dx_ball, dy_ball, + one-hot features
EDGE_INPUT_DIM = 4   # distance, angle, rel_vx, rel_vy

print("Building model...")
print("-" * 60)

    node_input_dim=NODE_INPUT_DIM,
    edge_input_dim=EDGE_INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    num_graph_layers=NUM_GRAPH_LAYERS,
    num_temporal_layers=NUM_TEMPORAL_LAYERS,
    num_heads=NUM_HEADS,
    max_trajectory_length=MAX_TRAJECTORY_LENGTH,
    dropout=DROPOUT,
    use_evoformer=True
)

# Move to device
model = model.to(device)

# Multi-GPU support
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print()


IndentationError: unexpected indent (1117933140.py, line 13)

## 5. Setup Training


In [ ]:
# Training hyperparameters
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 50
ACCELERATION_WEIGHT = 0.1  # Weight for acceleration smoothness loss

# Optimizer
optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS,
    eta_min=1e-6
)

# Loss function
def compute_loss(pred, target, mask):
    """
    Compute MSE loss with acceleration smoothness.
    
    Args:
        pred: [batch, seq_len, 2]
        target: [batch, seq_len, 2]
        mask: [batch, seq_len] - True for valid positions
    """
    # MSE loss (only on valid positions)
    mse_loss = ((pred - target) ** 2 * mask.unsqueeze(-1)).sum() / mask.sum()
    
    # Acceleration smoothness (penalize large changes in velocity)
    pred_vel = pred[:, 1:] - pred[:, :-1]  # [batch, seq_len-1, 2]
    pred_acc = pred_vel[:, 1:] - pred_vel[:, :-1]  # [batch, seq_len-2, 2]
    
    target_vel = target[:, 1:] - target[:, :-1]
    target_acc = target_vel[:, 1:] - target_vel[:, :-1]
    
    acc_mask = mask[:, 2:]  # Valid positions for acceleration
    acc_loss = ((pred_acc - target_acc) ** 2 * acc_mask.unsqueeze(-1)).sum() / acc_mask.sum()
    
    total_loss = mse_loss + ACCELERATION_WEIGHT * acc_loss
    
    return total_loss, mse_loss, acc_loss

print("Training setup complete!")
print(f"Learning rate: {LEARNING_RATE}")
print(f"Weight decay: {WEIGHT_DECAY}")
print(f"Epochs: {NUM_EPOCHS}")
print()


## 6. Training Loop


In [ ]:
# Training history
history = {
    'train_loss': [],
    'train_mse': [],
    'val_loss': [],
    'val_mse': [],
    'val_rmse': []
}

best_val_rmse = float('inf')
best_epoch = 0

print("Starting training...")
print("=" * 60)

for epoch in range(NUM_EPOCHS):
    # Training phase
    model.train()
    train_losses = []
    train_mses = []
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Train]")
    for batch in train_pbar:
        graph = batch["graph"].to(device)
        target = batch['target'].to(device)  # [batch, seq_len, 2]
        mask = batch["mask"].to(device)  # [batch, seq_len]
        
        # Move graph to device
        if hasattr(graph, 'to'):
            graph = batch["graph"].to(device)
        else:
            graph = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                    for k, v in graph.items()}
        
        # Forward pass
        optimizer.zero_grad()
        pred = model(graph)  # [batch, seq_len, 2]
        
        # Compute loss
        loss, mse_loss, acc_loss = compute_loss(pred, target, mask)
        
        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_losses.append(loss.item())
        train_mses.append(mse_loss.item())
        
        train_pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'mse': f'{mse_loss.item():.4f}'
        })
    
    # Validation phase
    model.eval()
    val_losses = []
    val_mses = []
    val_rmses = []
    
    with torch.no_grad():
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [Val]")
        for batch in val_pbar:
            graph = batch["graph"].to(device)
            target = batch['target'].to(device)
            mask = batch["mask"].to(device)
            
            # Move graph to device
            if hasattr(graph, 'to'):
                graph = batch["graph"].to(device)
            else:
                graph = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                        for k, v in graph.items()}
            
            # Forward pass
            pred = model(graph)
            
            # Compute loss
            loss, mse_loss, acc_loss = compute_loss(pred, target, mask)
            
            # Compute RMSE (only on valid positions)
            squared_diff = ((pred - target) ** 2 * mask.unsqueeze(-1))
            mse = squared_diff.sum() / mask.sum()
            rmse = torch.sqrt(mse)
            
            val_losses.append(loss.item())
            val_mses.append(mse_loss.item())
            val_rmses.append(rmse.item())
            
            val_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'rmse': f'{rmse.item():.4f}'
            })
    
    # Update history
    avg_train_loss = np.mean(train_losses)
    avg_train_mse = np.mean(train_mses)
    avg_val_loss = np.mean(val_losses)
    avg_val_mse = np.mean(val_mses)
    avg_val_rmse = np.mean(val_rmses)
    
    history['train_loss'].append(avg_train_loss)
    history['train_mse'].append(avg_train_mse)
    history['val_loss'].append(avg_val_loss)
    history['val_mse'].append(avg_val_mse)
    history['val_rmse'].append(avg_val_rmse)
    
    # Update scheduler
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    # Print epoch summary
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print(f"  Train Loss: {avg_train_loss:.4f} | Train MSE: {avg_train_mse:.4f}")
    print(f"  Val Loss: {avg_val_loss:.4f} | Val MSE: {avg_val_mse:.4f} | Val RMSE: {avg_val_rmse:.4f}")
    print(f"  LR: {current_lr:.6f}")
    
    # Save best model
    if avg_val_rmse < best_val_rmse:
        best_val_rmse = avg_val_rmse
        best_epoch = epoch + 1
        
        # Save model
        model_path = config.PROJECT_ROOT / "models" / "stgnn_transformer.pt"
        model_path.parent.mkdir(exist_ok=True)
        
        model_to_save = model.module if hasattr(model, 'module') else model
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model_to_save.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_rmse': avg_val_rmse,
            'history': history,
            'config': {
                'hidden_dim': HIDDEN_DIM,
                'num_graph_layers': NUM_GRAPH_LAYERS,
                'num_temporal_layers': NUM_TEMPORAL_LAYERS,
                'num_heads': NUM_HEADS,
                'max_trajectory_length': MAX_TRAJECTORY_LENGTH,
            }
        }, model_path)
        print(f"  ✓ Saved best model (RMSE: {best_val_rmse:.4f})")
    
    print("-" * 60)

print(f"\nTraining complete!")
print(f"Best validation RMSE: {best_val_rmse:.4f} (epoch {best_epoch})")
print()


In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MSE
axes[1].plot(history['train_mse'], label='Train MSE', linewidth=2)
axes[1].plot(history['val_mse'], label='Val MSE', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE')
axes[1].set_title('Mean Squared Error')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# RMSE
axes[2].plot(history['val_rmse'], label='Val RMSE', linewidth=2, color='green')
axes[2].axhline(y=best_val_rmse, color='r', linestyle='--', label=f'Best: {best_val_rmse:.4f}')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('RMSE')
axes[2].set_title('Validation RMSE')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. Sample Predictions


In [ ]:
# Load best model
model_path = config.PROJECT_ROOT / "models" / "stgnn_transformer.pt"
checkpoint = torch.load(model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Get a few validation samples
num_samples = 4
sample_indices = np.random.choice(len(val_dataset), num_samples, replace=False)

fig, axes = plt.subplots(2, 2, figsize=(16, 16))
axes = axes.flatten()

with torch.no_grad():
    for idx, ax in zip(sample_indices, axes):
        sample = val_dataset[idx]
        graph = sample['graph']
        target = sample['target']  # [seq_len, 2]
        mask = sample['mask']  # [seq_len]
        key = sample['key']
        traj_len = sample['trajectory_length']
        
        # Move to device
        if hasattr(graph, 'to'):
            graph = batch["graph"].to(device)
        else:
            graph = {k: v.to(device) if isinstance(v, torch.Tensor) else v 
                    for k, v in graph.items()}
        
        # Predict
        pred = model(graph)  # [1, seq_len, 2]
        pred = pred[0].cpu().numpy()  # [seq_len, 2]
        target = target.numpy()
        mask = mask.numpy()
        
        # Plot
        valid_mask = mask[:traj_len]
        ax.plot(target[:traj_len, 0], target[:traj_len, 1], 
               'b-o', label='True', markersize=4, linewidth=2, alpha=0.7)
        ax.plot(pred[:traj_len, 0], pred[:traj_len, 1], 
               'r--s', label='Predicted', markersize=4, linewidth=2, alpha=0.7)
        
        # Start position
        ax.plot(target[0, 0], target[0, 1], 'go', markersize=10, label='Start', zorder=5)
        
        ax.set_xlabel('X Position (yards)')
        ax.set_ylabel('Y Position (yards)')
        ax.set_title(f'Game {key[0]}, Play {key[1]}, Player {key[2]}')
        ax.legend()
        ax.grid(True, alpha=0.3)
        ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.show()


## Summary

Training complete! 

**Best Validation RMSE:** {best_val_rmse:.4f} (epoch {best_epoch})

**Model saved to:** `models/stgnn_transformer.pt`

The model is ready for inference and submission.
